# Does Width or Depth Matter More for a Reconstructor?

`CNNSizeComparison.ipynb` asked how much a fixed architecture's *size* matters, sweeping `ClassicCNN`'s `base_channels` across roughly 100k-3M trainable parameters. This notebook asks a sharper question at each of three target budgets (~300k, ~1M, ~3M trainable parameters): for a *fixed* parameter budget, is it better to spend it on width (more channels per layer, shallow) or depth (more layers, narrow)? A third variant at each budget adds residual (skip) connections to the deep network, since a plain deep conv stack is a classic vanishing-gradient risk that could otherwise confound "depth is worse" with "this particular deep network was just hard to optimize."

Nine reconstructors are built from one shared, parametrized `FlexCNN` family (`base_channels`, `blocks_per_stage`, and a plain-vs-residual block type are the only three knobs) -- 3 target sizes x {wide, deep, deep+residual} -- trained under otherwise identical conditions (same fictional-demo Pyramid WFS + DM, same loss, same `Trainer` procedure as `Tutorials/basics/05_TrainingAReconstructor.ipynb`/`CNNSizeComparison.ipynb`) and compared on training-loss curves and seeded closed-loop steady-state residual wavefront error (nm RMS).

This notebook reads `CNNComparison_params.py`, the single params file shared by every comparison notebook in this folder (same fictional Pyramid WFS/DM instrument, `Nout=42`/`Nmodes=190`), so results here sit on the identical instrument as every other comparison, including `CNNSizeComparison.ipynb`'s size sweep. Checkpoints are still kept in their own `Data/WidthDepthComparison/` directory, since these are a different set of trained weights.

Same deliberate trade-off as every other notebook in this series: `Trainer.train()` redraws a fresh, independent atmosphere at every outer step (`PhaseDataset.__getitem__`'s `idx == 0` branch always redraws), so training the nine architectures one after another on the same shared `dataset` object gives each its own independent i.i.d. stream rather than a paired one -- the nine training-loss curves below are directionally informative rather than a rigorously paired test. Only the seeded closed-loop rollout further down is a true apples-to-apples comparison.

In [ ]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os

from AI4AO.paths import DATA_DIR
from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, Trainer, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss, Physics_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'


## Configuration and instrument

Reuses `CNNComparison_params.py` unchanged -- the single params file shared by every comparison notebook in this folder -- so all of them compare on the identical instrument. `wfs`/`dm` are built fresh and frozen (`.eval()`); only each variant's reconstructor weights are trained.

In [ ]:
paramfile = 'CNNComparison_params.py'

AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

PATH = DATA_DIR / "WidthDepthComparison"
os.makedirs(PATH, exist_ok=True)

wfs = PyramidWFS(WFSParams, device)
wfs.eval()

dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.generateClosedLoop = True

M2C = dm.MakeZernikeM2C()
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

loss = LogResidualVarianceLoss(dataset.pupil, wfs.wavelength) + Physics_loss(wfs=wfs)

n_channels = wfs.pupil_centers.shape[0]
Nout = framePreprocessor.Nout
Nmodes = DMParams["Nmodes"]
print(f"Pyramid pupils: {n_channels}, preprocessed pupil size: {Nout}x{Nout}, Nmodes: {Nmodes}")


## One shared `FlexCNN` family: width, depth, and depth-with-residuals

Every variant below shares the exact same skeleton as `CNNSizeComparison.ipynb`'s `ClassicCNN`: `num_downsamples(Nout)` VGG-style downsampling stages (4 at this instrument's `Nout=42`, since the pupil crop's resolution -- not any architectural choice -- caps how many times a `42x42` feature map can be halved before dropping below `min_size=2`), followed by global-average-pool + one `Linear` head. `FlexCNN` factors that skeleton into three independent knobs so *how* each of the four stages spends its parameters is the only thing that varies between architectures:

- `base_channels` -- the width of the first stage (doubling every stage after, same as `ClassicCNN`).
- `blocks_per_stage` -- how many conv blocks are stacked *within* each of the 4 stages before its `MaxPool2d`. `blocks_per_stage=1` reproduces `ClassicCNN`'s stage exactly (its original two convs). Since the 4-stage depth is capped by resolution, this is the only way left to add real depth to the network.
- `block_cls` -- `PlainBlock` (two plain `3x3` convs) or `ResidualBlock` (the same two convs, plus a skip connection added before the block's final activation, with a `1x1` projection conv only where a stage's channel count changes, i.e. only on each stage's first block).

Three variants come out of this at each target parameter budget: **wide** (`blocks_per_stage=1`, large `base_channels`), **deep** (`blocks_per_stage` in the 5-6 range, small `base_channels`, `PlainBlock`), and **deep+residual** (same shape as deep, `ResidualBlock` instead) -- isolating "does depth need residual connections to pay off" as a controlled third arm rather than only comparing width against one specific deep architecture.

In [ ]:
def num_downsamples(size, min_size=2):
    """How many stride-2 poolings a feature map of this spatial size can
    take before dropping below min_size pixels -- same helper as
    CNNArchitectureComparison.ipynb's ClassicCNN, reused unmodified here."""
    n = 0
    while size // 2 >= min_size:
        size //= 2
        n += 1
    return n


class PlainBlock(nn.Module):
    """Two plain 3x3 convs -- one 'unit' of depth. blocks_per_stage=1 with
    this block reproduces ClassicCNN's stage exactly."""

    def __init__(self, in_channels, out_channels, activation=nn.LeakyReLU):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act1 = activation()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act2 = activation()

    def forward(self, x):
        x = self.act1(self.conv1(x))
        return self.act2(self.conv2(x))


class ResidualBlock(nn.Module):
    """Same two 3x3 convs as PlainBlock, plus a skip connection added before
    the final activation. A 1x1 projection conv is only needed on a stage's
    first block, where in_channels != out_channels; every later block in the
    stage uses a plain identity skip."""

    def __init__(self, in_channels, out_channels, activation=nn.LeakyReLU):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act1 = activation()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.skip = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )
        self.act2 = activation()

    def forward(self, x):
        identity = self.skip(x)
        out = self.conv2(self.act1(self.conv1(x)))
        return self.act2(out + identity)


def conv_stage(in_channels, out_channels, n_blocks, block_cls, activation=nn.LeakyReLU):
    """n_blocks copies of block_cls (only the first taking in_channels ->
    out_channels, the rest out_channels -> out_channels), then a 2x
    downsampling MaxPool -- one stage."""
    layers = []
    for b in range(n_blocks):
        block_in = in_channels if b == 0 else out_channels
        layers.append(block_cls(block_in, out_channels, activation))
    layers.append(nn.MaxPool2d(2))
    return layers


class FlexCNN(nn.Module):
    """ClassicCNN's skeleton (num_downsamples(Nout) stages, global-average-pool
    + Linear head) with base_channels, blocks_per_stage, and block_cls factored
    out as independent knobs -- base_channels=X, blocks_per_stage=1, block_cls=
    PlainBlock reproduces ClassicCNN exactly."""

    def __init__(self, n_channels, Nmodes, Nout, base_channels, blocks_per_stage, block_cls):
        super().__init__()

        n_stages = num_downsamples(Nout)
        channels = [n_channels] + [base_channels * 2 ** i for i in range(n_stages)]

        layers = []
        for i in range(n_stages):
            layers += conv_stage(channels[i], channels[i + 1], blocks_per_stage, block_cls)
        layers.append(nn.AdaptiveAvgPool2d(1))

        self.encoder = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(channels[-1], Nmodes))

    def forward(self, x):
        x = self.encoder(x)
        return self.head(x)


## Nine architectures: 3 target sizes x {wide, deep, deep+residual}

`(base_channels, blocks_per_stage)` for each of the nine cells below were picked by an actual parameter-count sweep against this instrument's `Nout=42`/`Nmodes=190` (not hand arithmetic), the same way `CNNSizeComparison.ipynb`'s `BASE_CHANNELS_LIST` documents its own sweep. The wide variant at each budget is exactly `ClassicCNN` (`blocks_per_stage=1`) at the `base_channels` that lands closest to the target; the deep/deep+residual variants hold `base_channels` at roughly 40% of the wide variant's (so they are meaningfully narrower, not just "wide with a couple more layers") and use `blocks_per_stage` in the 5-6 range to make up the rest of the same parameter budget -- roughly 40-48 conv layers deep vs. the wide variant's 8.

In [ ]:
VARIANT_STYLE = {
    # Fixed color per variant (never repainted across plots) + a distinct
    # marker/linestyle per variant, consistent everywhere in this notebook.
    "wide": dict(color="#E69F00", marker="o", linestyle="-", label="Wide (blocks_per_stage=1)"),
    "deep": dict(color="#56B4E9", marker="s", linestyle="--", label="Deep (PlainBlock)"),
    "deep_res": dict(color="#009E73", marker="^", linestyle=":", label="Deep + residual (ResidualBlock)"),
}

SIZE_CONFIGS = {
    "300k": {
        "wide": dict(base_channels=16, blocks_per_stage=1, block_cls=PlainBlock),
        "deep": dict(base_channels=6, blocks_per_stage=6, block_cls=PlainBlock),
        "deep_res": dict(base_channels=6, blocks_per_stage=5, block_cls=ResidualBlock),
    },
    "1M": {
        "wide": dict(base_channels=29, blocks_per_stage=1, block_cls=PlainBlock),
        "deep": dict(base_channels=12, blocks_per_stage=5, block_cls=PlainBlock),
        "deep_res": dict(base_channels=12, blocks_per_stage=5, block_cls=ResidualBlock),
    },
    "3M": {
        "wide": dict(base_channels=51, blocks_per_stage=1, block_cls=PlainBlock),
        "deep": dict(base_channels=20, blocks_per_stage=5, block_cls=PlainBlock),
        "deep_res": dict(base_channels=20, blocks_per_stage=5, block_cls=ResidualBlock),
    },
}

models = {}
for size_name, variants in SIZE_CONFIGS.items():
    for variant_name, cfg in variants.items():
        name = f"{size_name}_{variant_name}"
        model = FlexCNN(n_channels, Nmodes, Nout, **cfg).to(device=device)
        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        models[name] = model
        print(f"{name:12s} -- total trainable parameters: {total_params:,}")


## Training

`05_TrainingAReconstructor.ipynb`'s exact procedure, repeated once per architecture: a fresh `AdamW` optimizer and a fresh `Trainer` per architecture, all nine sharing the same `wfs`/`dm`/`framePreprocessor`/`dataset`/`loss` objects, each calling `trainer.train(TrainRunNb, ClosedLoopIterations)` for the same number of steps regardless of size or shape -- the question this notebook asks is what a fixed training budget buys under each allocation of capacity, not whether every architecture can eventually be trained to convergence given unlimited steps.

Nine full training runs is close to double `CNNSizeComparison.ipynb`'s five -- a real wall-clock cost on the single memory-constrained GPU CLAUDE.md assumes. Consider dropping `TrainRunNb` for an initial correctness pass (confirm every shape/forward pass works, especially the residual variants) before committing to the full budget below.

In [ ]:
models

In [ ]:
# Closed-loop (BPTT) iterations per optimizer step in the second training stage. Stage 1 uses
# TrainParams['ClosedLoopIterations'] from CNNComparison_params.py (a single pass); this is the
# one training setting that is not in that shared file, so it lives here.
CLOSED_LOOP_ITERATIONS_STAGE2 = 10

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in models.items():
    print(f"=== Training {name} (stage 1: single pass) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by a
        # different SIZE_CONFIGS choice.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], TrainParams['ClosedLoopIterations'])

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


### Stage 2 -- closed-loop training

The loop above (stage 1) trains every model with `TrainParams['ClosedLoopIterations']` from `CNNComparison_params.py`, i.e. a single pass per optimizer step. The next cell repeats the identical loop with `CLOSED_LOOP_ITERATIONS_STAGE2` closed-loop iterations per optimizer step, continuing from the weights stage 1 just saved (`load_checkpoint` at the top of the loop, and a fresh `AdamW` per model, as in stage 1).

The checkpoint under `PATH` is overwritten with the stage-2 weights, and from here on `trainers`/`loss_trackers`/`loss_trackers_ideal` -- the ones the plotting and evaluation cells below use -- refer to stage 2.

In [ ]:
trainers = {}
loss_trackers = {}
loss_trackers_ideal = {}

for name, model in models.items():
    print(f"=== Training {name} (stage 2: closed loop) ===")

    optimizer = torch.optim.AdamW(model.parameters(), TrainParams['lrn'], fused=(device == 'cuda'))

    trainer = Trainer(
        wfs=wfs,
        framePreprocessor=framePreprocessor,
        dm=dm,
        M2C=M2C,
        phaseReconstructor=model,
        dataset=dataset,
        loss=loss,
        optimizer=optimizer,
    )

    try:
        trainer.load_checkpoint(PATH / f"{name}.pth", load_optimizer=False)
    except (FileNotFoundError, RuntimeError):
        # RuntimeError covers a shape mismatch against a checkpoint saved by a
        # different SIZE_CONFIGS choice.
        print("Starting from scratch")

    loss_tracker, loss_tracker_ideal = trainer.train(TrainParams['TrainRunNb'], CLOSED_LOOP_ITERATIONS_STAGE2)

    trainer.save_checkpoint(PATH / f"{name}.pth")

    trainers[name] = trainer
    loss_trackers[name] = loss_tracker
    loss_trackers_ideal[name] = loss_tracker_ideal


## Comparing training-loss curves

`Trainer.plot_losses` only plots one tracker pair at a time, so all nine are overlaid manually here: color identifies the variant (wide/deep/deep+residual, fixed across every plot in this notebook), linestyle identifies the target size (solid=300k, dashed=1M, dotted=3M) -- so the same visual key applies everywhere below, not just here.

In [ ]:
def smooth(x, window=100):
    x = x.detach().cpu().numpy()
    return np.convolve(x, np.ones(window) / window, "valid")

SIZE_LINESTYLE = {"300k": "-", "1M": "--", "3M": ":"}

fig, ax = plt.subplots(figsize=(9, 6))
for size_name in SIZE_CONFIGS:
    for variant_name, style in VARIANT_STYLE.items():
        name = f"{size_name}_{variant_name}"
        ax.plot(
            smooth(loss_trackers[name]),
            color=style["color"],
            linestyle=SIZE_LINESTYLE[size_name],
            label=f"{style['label']}, {size_name}",
        )

first_ideal = next(iter(loss_trackers_ideal.values()))
ax.plot(smooth(first_ideal), label="Oracle bound (ideal)", color="black", linestyle="-", linewidth=2)

ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.legend(fontsize=8, ncol=2)
plt.show()


## Closed-loop residual wavefront error vs. parameter count

Same seeded-rollout protocol as `CNNSizeComparison.ipynb`: reseed immediately before each architecture's rollout so all nine see identical wind/r0/noise draws, then compute the steady-state residual wavefront error in nm RMS (excluding the leaky-integrator warm-up). Plotted against parameter count on a log-x axis, with the three width points, three deep points, and three deep+residual points each connected into their own line -- so the width-vs-depth-vs-depth+residual trend across all three budgets reads at a glance, not just budget by budget.

In [ ]:
def residual_nm_rms(result, pupil, warmup_fraction=0.3):
    """Steady-state residual wavefront error (nm RMS) over the pupil, from an
    EvaluationResult's residual_opd trajectory, excluding the leaky-integrator
    warm-up period (same 0.3 convention Trainer.evaluate() uses internally)."""
    n_steps = result.residual_opd.shape[0]
    warmup = int(n_steps * warmup_fraction)
    steady_state = result.residual_opd[warmup:]
    pupil_values = steady_state[..., pupil.bool()]
    return torch.sqrt(torch.mean(pupil_values ** 2)).item() * 1e9


eval_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset.generateClosedLoop = True

nm_rms = {}
param_counts = {}
for name, trainer in trainers.items():
    torch.manual_seed(TrainParams['Seed'])
    result = trainer.evaluate(n_steps=TrainParams['TestRunNb'], dataset=eval_dataset)
    nm_rms[name] = residual_nm_rms(result, dataset.pupil)
    param_counts[name] = sum(p.numel() for p in models[name].parameters() if p.requires_grad)
    print(f"{name:12s} -- {param_counts[name]:>9,} params -- steady-state residual: {nm_rms[name]:.1f} nm RMS")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for variant_name, style in VARIANT_STYLE.items():
    xs = [param_counts[f"{size_name}_{variant_name}"] for size_name in SIZE_CONFIGS]
    ys = [nm_rms[f"{size_name}_{variant_name}"] for size_name in SIZE_CONFIGS]
    order = np.argsort(xs)
    xs = [xs[i] for i in order]
    ys = [ys[i] for i in order]
    ax.plot(xs, ys, marker=style["marker"], color=style["color"], linestyle=style["linestyle"], label=style["label"])

ax.set_xscale("log")
ax.set_xlabel("Trainable parameters")
ax.set_ylabel("Steady-state residual (nm RMS)")
ax.set_title(f"Width vs. depth vs. depth+residual (fixed {TrainParams['TrainRunNb']}-step training budget)")
ax.legend()
plt.tight_layout()
plt.show()


## Visualizing the best architecture's closed loop

A closer look at whichever of the nine architectures came out on top above -- the same rollout-animation pattern `05_TrainingAReconstructor.ipynb`/`CNNSizeComparison.ipynb` use.

In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

best_name = min(nm_rms, key=nm_rms.get)
print(f"Best architecture: {best_name} ({nm_rms[best_name]:.1f} nm RMS)")

n_frames = 60
torch.manual_seed(TrainParams['Seed'])
result = trainers[best_name].evaluate(n_steps=n_frames, dataset=eval_dataset)

fig, axes = imshow_multiple(
    [
        {"tensor": result.opd[0], "title": "Input OPD", "same_scale": True},
        {"tensor": result.residual_opd[0], "title": "Residual OPD", "scale_reference": result.opd[0]},
        {"tensor": result.wfs_frames[0], "title": "WFS frame"},
    ],
    max_channel_number=9,
)


def update(i):
    imshow_multiple(
        [
            {"tensor": result.opd[i], "title": "Input OPD", "same_scale": True},
            {"tensor": result.residual_opd[i], "title": "Residual OPD", "scale_reference": result.opd[i]},
            {"tensor": result.wfs_frames[i], "title": "WFS frame"},
        ],
        fig=fig, axes=axes,
        max_channel_number=9,
    )
    return [ax.images[0] for tensor_axes in axes for ax in tensor_axes]


anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
plt.close(fig)

mpl.rcParams["animation.embed_limit"] = 200
HTML(anim.to_jshtml())
